# Combat Critic → ONNX (Colab tip #3)

Export tip #2 **`combat_critic_smoke.pt`** to ONNX; verify PyTorch vs **onnxruntime** (max abs err ≤ **1e-4**, fp32).

**I/O names:** `obs` `(N, 181)` float32 → `value` `(N, 1)` float32.

No game comms / EP / eval suite. Run tip #2 first so Drive has the `.pt` checkpoint.

In [ ]:
# !git clone --depth 1 https://github.com/EienteiPharma/sts2-rl-agent.git /content/sts2-rl-agent
!pip install -q numpy torch onnx onnxruntime

import sys
from pathlib import Path

REPO = Path("/content/sts2-rl-agent")
if not REPO.is_dir():
    raise SystemExit("Clone repo to /content/sts2-rl-agent first (uncomment git clone).")
sys.path.insert(0, str(REPO))

from sts2_env.colab.combat_critic import export_critic_onnx, verify_critic_onnx

In [ ]:
from google.colab import drive  # type: ignore
from pathlib import Path

USE_DRIVE = True  # False → /content/sts2/colab/ fallback
DRIVE_COLAB_DIR = "/content/drive/MyDrive/sts2/colab"
LOCAL_COLAB_DIR = "/content/sts2/colab"

CKPT_PATH = f"{DRIVE_COLAB_DIR}/combat_critic_smoke.pt"
ONNX_PATH = f"{DRIVE_COLAB_DIR}/combat_critic_smoke.onnx"

if USE_DRIVE:
    drive.mount("/content/drive")
    Path(DRIVE_COLAB_DIR).mkdir(parents=True, exist_ok=True)
else:
    Path(LOCAL_COLAB_DIR).mkdir(parents=True, exist_ok=True)
    CKPT_PATH = f"{LOCAL_COLAB_DIR}/combat_critic_smoke.pt"
    ONNX_PATH = f"{LOCAL_COLAB_DIR}/combat_critic_smoke.onnx"

print("CKPT_PATH", CKPT_PATH)
print("ONNX_PATH", ONNX_PATH)

In [ ]:
from pathlib import Path

if not Path(CKPT_PATH).is_file():
    raise SystemExit(f"Missing checkpoint: {CKPT_PATH} (run tip #2 first)")

onnx_out = export_critic_onnx(CKPT_PATH, ONNX_PATH)
verify = verify_critic_onnx(CKPT_PATH, onnx_out, batch_size=32, seed=0)
print("ONNX_EXPORT_PASS", onnx_out, verify["max_abs_err"])
verify